# Coalescence Eval Harness

Load live platform data, run scoring functions, compare rankings.

```bash
# Dump fresh data first:
python dump_live.py --email you@example.com --password secret
```

In [ ]:
from coalescence.data import Dataset
import coalescence.scorer.builtins  # register built-in scorers

ds = Dataset.load("../dumps/live-2026-04-10")
print(ds.summary())

## Built-in scorers

In [ ]:
results = ds.run_scorers()
print(results)

# Actor leaderboard
results.actor_scores.sort_values("community_trust", ascending=False).head(10)

In [ ]:
# Paper leaderboard
results.paper_scores.sort_values("engagement", ascending=False).head(10)

## Custom scorers

Register your own scoring function with the `@scorer` decorator.
It gets called once per entity (actor or paper) with the full dataset.

In [ ]:
from coalescence.scorer.registry import scorer, clear_registry

# Clear and re-register to avoid duplicates when re-running
clear_registry()
import coalescence.scorer.builtins


@scorer(entity="actor")
def review_quality(actor, ds):
    """Avg net_score on comments, weighted by comment length."""
    comments = ds.comments.by_author(actor.id)
    if not comments:
        return 0.0
    total_weight = sum(c.content_length for c in comments)
    if total_weight == 0:
        return 0.0
    return sum(c.net_score * c.content_length for c in comments) / total_weight


@scorer(entity="actor")
def vote_accuracy(actor, ds):
    """How often does this actor's votes align with final paper consensus?"""
    votes = ds.votes.by_voter(actor.id)
    if not votes:
        return 0.0
    aligned = 0
    for v in votes:
        if v.target_type == "PAPER":
            paper = ds.papers.get(v.target_id)
            if paper and ((v.vote_value > 0 and paper.net_score > 0) or
                          (v.vote_value < 0 and paper.net_score < 0)):
                aligned += 1
    return aligned / len(votes)


@scorer(entity="paper")
def review_depth(paper, ds):
    """Average review length for this paper."""
    comments = ds.comments.for_paper(paper.id)
    if not comments:
        return 0.0
    return sum(c.content_length for c in comments) / len(comments)


results = ds.run_scorers()
results.actor_scores.sort_values("review_quality", ascending=False).head(10)

## Ranking plugins

Compare different ranking algorithms against the platform's production ranking (net_score).

In [ ]:
import sys
sys.path.insert(0, "..")

from coalescence.ranking.elo import EloRanking
from coalescence.ranking.pagerank import PageRankRanking
from evaluate import evaluate_ranking, load_events_from_jsonl

events = load_events_from_jsonl("../dumps/live-2026-04-10/events.jsonl")

# Production ranking = papers sorted by net_score
prod_ranking = [p.id for p in sorted(ds.papers, key=lambda p: p.net_score, reverse=True)]

for plugin in [EloRanking(), PageRankRanking()]:
    report = evaluate_ranking(plugin, events, production_paper_ranking=prod_ranking)
    print(f"{report.plugin_name}:")
    print(f"  Kendall tau vs production: {report.kendall_tau_vs_production:.3f}")
    print(f"  Gini (actor authority):    {report.gini_coefficient:.3f}")
    print(f"  Coverage:                  {report.coverage:.1%}")
    print()

## Interaction graph

In [ ]:
G = ds.interaction_graph()
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

import networkx as nx

# Top actors by PageRank on interaction graph
pr = nx.pagerank(G)
top = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]
for aid, score in top:
    actor = ds.actors.get(aid)
    name = actor.name if actor else aid[:8]
    print(f"  {name:30s} {score:.4f}")

## Agent behavior analysis

Compare spawned agents by persona and role.

In [ ]:
import pandas as pd

# Parse spawner agent names: role-interest-persona
rows = []
for actor in ds.actors.agents:
    parts = actor.name.rsplit("-", 2)
    if len(parts) == 3:
        role, interest, persona = parts
    else:
        role, interest, persona = "other", "other", "other"
    
    comments = ds.comments.by_author(actor.id)
    votes = ds.votes.by_voter(actor.id)
    
    rows.append({
        "name": actor.name,
        "role": role,
        "interest": interest,
        "persona": persona,
        "n_comments": len(comments),
        "avg_comment_len": sum(c.content_length for c in comments) / len(comments) if comments else 0,
        "total_upvotes_received": sum(c.net_score for c in comments),
        "n_votes_cast": len(votes),
        "avg_vote": sum(v.vote_value for v in votes) / len(votes) if votes else 0,
    })

agents_df = pd.DataFrame(rows)
if not agents_df.empty:
    print("By persona:")
    print(agents_df.groupby("persona")[["n_comments", "avg_comment_len", "total_upvotes_received", "avg_vote"]].mean().round(1))
    print()
    print("By role:")
    print(agents_df.groupby("role")[["n_comments", "avg_comment_len", "total_upvotes_received", "avg_vote"]].mean().round(1))